In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
import sys

#math and array operations
import numpy as np
import math
import pandas as pd

#data classes
import xarray as xr
import pickle

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#loading bar
from tqdm import tqdm

#datetime
from datetime import datetime

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "Observation_Data")
dataType = "RadarComparison"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
def GetSimulationTime(RunType):
    if (RunType[0] == "TRACER") and (RunType[1] == "MOIST"):
        SimulationTime = ("2022-06-30","2022-07-03")
    elif (RunType[0] == "TRACER") and (RunType[1] == "DRY"):
        SimulationTime = ("2022-06-08","2022-06-11")
    return SimulationTime

# spinup_hours = "24"
# spinup_hours = "12"
spinup_hours = "6"

RunType = ("TRACER","MOIST","NSSL",spinup_hours)
# RunType = ("TRACER","DRY","NSSL",spinup_hours)
SimulationTime = GetSimulationTime(RunType)
ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType, SimulationTime)

RunType = ("TRACER","MOIST","TEMPO",spinup_hours)
# RunType = ("TRACER","DRY","TEMPO",spinup_hours)
SimulationTime = GetSimulationTime(RunType)
ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType, SimulationTime)

In [ ]:
#Importing Radar Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class

In [ ]:
#Importing ERA5 Data Loading Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","ERA5_Data"))
from CLASSES_ERA5DataLoading import ERA5DataLoading_Class,ERA5DataLoading_Class_gdex

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSaving import DataSaving_Class

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Plotting import ContourPlotting_Class

In [ ]:
########################
#DATA INFORMATION

In [ ]:
#DATA CITATION
# Atmospheric Radiation Measurement (ARM) user facility. 2021. X-Band Scanning ARM Cloud Radar (XSACRCFRQC), 2022-06-09 to 2022-07-02, ARM Mobile Facility (HOU) Houston, TX; AMF1 (main site for TRACER) (M1). Compiled by Y. Feng, A. Matthews, E. Schuman, K. Johnson, I. Lindenmaier, V. Castro and T. Wendler. ARM Data Center. Data set accessed 2025-11-05 at http://dx.doi.org/10.5439/2001296.

#Globus Download Link
# https://urldefense.com/v3/__https://app.globus.org/file-manager?origin_id=ba87aabe-30f6-433d-b4a5-19434c595e0f&origin_path=*rosemana1*261781*__;Ly8v!!PvDODwlR4mBZyAb0!REKAOzHJNvJk50CY5Pjl135CV83BhArwtdyMDuBM-28KqreBug8Xb5Mc3MLgy_p9PUiWe2uVXq-EfUtLcGdH5w$

In [ ]:
#DATA CITATION
# Atmospheric Radiation Measurement (ARM) user facility. 2021. Ka-Band Scanning ARM Cloud Radar (KASACRCFRQC), 2022-06-08 to 2022-07-02, ARM Mobile Facility (HOU) Houston, TX; AMF1 (main site for TRACER) (M1). Compiled by I. Lindenmaier, K. Johnson, D. Nelson, A. Matthews, T. Wendler, V. Melo de Castro, M. Rocque and Y. Feng. ARM Data Center. Data set accessed 2025-11-05 at http://dx.doi.org/10.5439/1877338.

# https://armgov.svcs.arm.gov/capabilities/instruments/kasacr

#Globus Download Link
#https://urldefense.com/v3/__https://app.globus.org/file-manager?origin_id=ba87aabe-30f6-433d-b4a5-19434c595e0f&origin_path=*rosemana1*261893*__;Ly8v!!PvDODwlR4mBZyAb0!UJrKWg_xaYuaaa_iaY91TCVMB_neSskXsDKHtPPCG7Ix6sEmiEvnngTUrYuV18LudZqxB3eHyTDuCHSZ3o3zrg$

In [ ]:
#LOADING RADAR CLASS
RadarData_MRMS = RadarData_MRMS_Class(ModelData_NSSL,
                                      fileDirectory=os.path.join(DirectoryManager.dataDirectory,
                                                                 "Observation_Data/TRACER/MRMS_RadarData",
                                                                 f"{ModelData_NSSL.simulationDates[0]}_{ModelData_NSSL.simulationDates[-1]}"))

In [ ]:
##########################
#DATA LOADING FUNCTIONS

In [ ]:
#Converting timeStrings
def ConvertTimeStringtoDateTime(timeString):
    """
    Converts a time string like '2022-06-30_00.00.00' to a datetime object.
    """
    return datetime.strptime(timeString, '%Y-%m-%d_%H.%M.%S')
def ConvertTimeStringtoTimeTitle(timeString):
    """
    Converts a time string like '2022-06-30_00.00.00' to a datetime object.
    Formatted for use as a plot title.
    """
    # Parse the custom format to a datetime object
    dt = datetime.strptime(timeString, '%Y-%m-%d_%H.%M.%S')
    # Format it to 'YYYY-MM-DD HH:MM:SS'
    return dt.strftime('%Y-%m-%d %H:%M:%S')

In [ ]:
#Getting TimeData
def GetData(t):
    timeString = ModelData_NSSL.timeStrings[t]
    timeString_datetime = ConvertTimeStringtoDateTime(timeString)
    
    #Loading Model Radar
    modelRadarData_NSSL = ModelData_NSSL.GetDataTimestep_diag(t)["refl10cm_1km"]
    modelRadarData_TEMPO = ModelData_TEMPO.GetDataTimestep_diag(t)["refl10cm_1km"]
    modelRadarTimeTitle = ConvertTimeStringtoTimeTitle(timeString)
    
    #Loading Observational Radar
    radarData, nearestFilePath = RadarData_MRMS.LoadClosestMRMSFile(target_time=timeString_datetime)
    radarTimeTitle = pd.to_datetime(radarData['time'].data[0]).strftime("%Y-%m-%d %H:%M:%S")
    radarData=radarData.isel(time=0)

    #Getting Model MSLP Data
    mslpData_NSSL = ModelData_NSSL.GetDataTimestep_diag(t)['mslp']/1e2
    mslpData_TEMPO = ModelData_TEMPO.GetDataTimestep_diag(t)['mslp']/1e2

    #Getting ERA5 MSLP Data
    mslp_ERA5_alltimes = ERA5DataLoading_Class_gdex.LoadERA5Data(timeString, ModelData_NSSL, DirectoryManager)
    mslp_ERA5 = ERA5DataLoading_Class_gdex.SelectNearestERA5Time(mslp_ERA5_alltimes, timeString)/1e2
    # mslp_ERA5_alltimes = ERA5DataLoading_Class.LoadERA5Data(DirectoryManager, ModelData_NSSL, variableName='msl',dataType='Surface')
    # mslp_ERA5 = ERA5DataLoading_Class.SelectNearestERA5Time(mslp_ERA5_alltimes, ModelData_NSSL.timeStrings[t])/1e2

    return (modelRadarData_NSSL,modelRadarData_TEMPO,modelRadarTimeTitle, 
            radarData,radarTimeTitle, 
            timeString,
            mslpData_NSSL,mslpData_TEMPO,mslp_ERA5)

In [ ]:
def FixLatLon_RadarData(radarData):        
    radarData = radarData.isel(latitude=slice(None, None, -1))

    radarData = radarData.assign_coords(
        longitude=((radarData.longitude + 180) % 360) - 180
    )
    return radarData

def ReturnLatLon_RadarData(radarData):
    # Fix latitude order
    radarData_fixed = radarData.isel(latitude=slice(None, None, -1))

    # Fix longitude convention
    radarData_fixed = radarData_fixed.assign_coords(
        longitude=radarData.longitude+360
    )

    return radarData_fixed


def InterpolateRadarData(radarData,modelData):
    radarData = FixLatLon_RadarData(radarData)
    
    radarData_interp = radarData.interp(
        latitude=modelData.latitude,
        longitude=modelData.longitude,
        method="linear"
    )
    return radarData_interp

In [ ]:
##########################
#PLOTTING FUNCTIONS

In [ ]:
def MakePlot(modelRadarData_NSSL,modelRadarData_TEMPO,modelRadarTimeTitle, 
             radarData,radarTimeTitle,
             mslpData_NSSL,mslpData_TEMPO,mslp_ERA5):
    
    fig, axes = RadarPlotting_Class.CreateMapAxes(nrows=1,ncols=3,
                                                  figsize=(16,8))
    
    #Plotting ModelRadar
    #nssl
    axis = axes[0,0]
    lat = modelRadarData_NSSL['latitude']
    lon = modelRadarData_NSSL['longitude']
    contourPlot = RadarPlotting_Class.PlotReflectivity(axis, lat,lon,modelRadarData_NSSL,dataName="NSSL",timeTitle=modelRadarTimeTitle)

    #Adding MSLP Contours
    cs1 = axis.contour(lon, lat, mslpData_NSSL, 
                       colors='black', levels=10, linewidths=1.0, alpha=0.35, zorder=10)
    labels = axis.clabel(cs1, inline=True, fontsize=8, fmt="%.0f",
                colors='black',zorder=11)
    for label in labels:
        label.set_alpha(1)
    
    #tempo
    axis = axes[0,2]
    lat = modelRadarData_TEMPO['latitude']
    lon = modelRadarData_TEMPO['longitude']
    RadarPlotting_Class.PlotReflectivity(axis, lat,lon,modelRadarData_TEMPO,dataName="TEMPO",timeTitle=modelRadarTimeTitle)

    #Adding MSLP Contours
    cs1 = axis.contour(lon, lat, mslpData_TEMPO, 
                       colors='black', levels=10, alpha=0.35, linewidths=1.0,zorder=10)
    labels = axis.clabel(cs1, inline=True, fontsize=8, fmt="%.0f",
                colors='black',zorder=11)
    for label in labels:
        label.set_alpha(1)
    
    #Plotting Observational Radar
    #mrms data
    axis = axes[0,1]
    lat = radarData['latitude'].data
    lon = radarData['longitude'].data-360
    
    RadarPlotting_Class.PlotReflectivity(axis, lat,lon,radarData,dataName="MRMS",timeTitle=radarTimeTitle)

    #Adding MSLP Contours
    cs1 = axis.contour(mslp_ERA5.longitude, mslp_ERA5.latitude, mslp_ERA5, 
                       colors='black', levels=10, alpha=0.35, linewidths=1.0,zorder=10)
    labels = axis.clabel(cs1, inline=True, fontsize=8, fmt="%.0f",
                colors='black',zorder=11)
    for label in labels:
        label.set_alpha(1)
    
    #Adding Colorbar
    colorBar = RadarPlotting_Class.AddSharedColorbar(fig, contourPlot)


    return fig

In [ ]:
##########################
# CALCULATING

In [ ]:
t=92
(modelRadarData_NSSL,modelRadarData_TEMPO,modelRadarTimeTitle, 
            radarData,radarTimeTitle, 
            timeString,
            mslpData_NSSL,mslpData_TEMPO,mslp_ERA5) = GetData(t)

radarData_interp = InterpolateRadarData(radarData=radarData, modelData=modelRadarData_NSSL)
radarData_interp = ReturnLatLon_RadarData(radarData_interp,) #not necessary unless plotting below

In [ ]:
##########################
#PLOTTING

In [ ]:
fig = MakePlot(modelRadarData_NSSL,modelRadarData_TEMPO,modelRadarTimeTitle, 
               radarData_interp,radarTimeTitle,
               mslpData_NSSL,mslpData_TEMPO,mslp_ERA5)

In [ ]:
##########################
#TESTING TOBAC

# https://tobac.readthedocs.io/en/latest/installation.html
# https://openradarscience.org/ams-open-radar-2023/notebooks/tobac/tobac_examples.html

In [ ]:
#Setup

In [ ]:
# mamba install -c conda-forge tobac
import tobac
import tobac.testing
import tobac.feature_detection
import tobac.segmentation

In [ ]:
feature_detection_params = dict()
feature_detection_params['threshold'] = [30]#, 40, 50]
feature_detection_params['target'] = 'maximum'
feature_detection_params['position_threshold'] = 'weighted_diff'
feature_detection_params['n_erosion_threshold'] = 2
feature_detection_params['sigma_threshold'] = 1
feature_detection_params['n_min_threshold'] = 4

In [ ]:
#Feature Detection

In [ ]:
xr_grid_full = modelRadarData_TEMPO

In [ ]:
grid_iris = xr_grid_full.to_iris()

In [ ]:
dxy, dt = 1000,15*60

In [ ]:
Features_df = tobac.feature_detection_multithreshold(data, dxy, **feature_detection_params)


In [ ]:
modelRadarData_TEMPO.plot()